# QMatBridge quickstart

Two short parts, both runnable offline from a checkout (no API key needed):

1. **Real entries.** Load the six Tier-1 entries that were fetched live from the Materials Project, inspect them as `QMatEntry` objects, see how the plane-wave count scales, and see what the fingerprint (`canonical_hash`) does and does not react to.
2. **A planning demo.** Rank three toy candidates with the example scripts' *proxy* cost model.

Part 1 is data; Part 2 is an illustration of a workflow, not a result.

In [ ]:
import copy
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import qmatbridge
from qmatbridge.basis import num_plane_waves_from_ecut
from qmatbridge.io import read_entry_json, write_entry_json

# Jupyter starts the kernel in the notebook's folder, so find the repo root by
# walking up from wherever we are instead of assuming the current directory.
start = Path.cwd().resolve()
repo_root = next(
    (
        p
        for p in (start, *start.parents)
        if (p / "pyproject.toml").is_file() and (p / "examples").is_dir()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError(
        f"Could not find the QMatBridge repo above {start}; "
        "open this notebook from inside a checkout."
    )
examples_dir = repo_root / "examples"
fixtures_dir = repo_root / "benchmarks" / "fixtures"
print(f"qmatbridge {qmatbridge.__version__}, repo root: {repo_root}")

## Part 1. Real entries

### Load the Tier-1 fixtures

Each fixture is a `QMatEntry` written by `write_entry_json` from a live Materials Project query. `read_entry_json` is strict: it rejects unknown fields, wrong types and non-finite numbers rather than guessing.

In [ ]:
KEYS = ["Si", "GaN", "LiCoO2", "LiFePO4", "NaCl", "BaTiO3"]
entries = {k: read_entry_json(fixtures_dir / f"{k}.json") for k in KEYS}

rows = []
for k, e in entries.items():
    s, h = e.reference.structure, e.hamiltonian
    rows.append({
        "material": k,
        "mp_id": e.reference.provenance.primary.identifier,
        "sites": s.num_sites,
        "space_group": f"{s.lattice.spacegroup_symbol} ({s.lattice.spacegroup_number})",
        "electrons": h.num_electrons,
        "functional": e.reference.provenance.functional,
        "cutoff_eV": h.basis.cutoff_energy_ev,
        "plane_waves": h.basis.num_plane_waves,
        "fingerprint": e.canonical_hash()[:12],
    })
table = pd.DataFrame(rows)
display(table)

`electrons` is the valence count from the VASP POTCARs the Materials Project used (for example Fe contributes 14, Li 3), and `plane_waves` is counted exactly at Γ for the recorded cutoff. Treat the plane-wave number as a reference scale taken from the DFT setup; a quantum algorithm may choose a different basis size.

### Size of the problem

The two numbers a first-quantized algorithm cares about most are the electron count η and the basis size N. They span a wide range across just six materials.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.scatter(table["electrons"], table["plane_waves"], s=70, color="#2a9d8f", edgecolors="#1b4332", zorder=3)
for _, r in table.iterrows():
    ax.annotate(r["material"], (r["electrons"], r["plane_waves"]), xytext=(6, 5), textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Valence electrons in the cell, $\\eta$")
ax.set_ylabel("Plane waves at $\\Gamma$, $N$")
ax.set_title("Tier-1 entries: electrons vs plane-wave basis size")
ax.grid(True, which="both", linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### How N depends on the cutoff

For a fixed cell, N grows as the cutoff to the power 3/2 (the number of reciprocal-lattice points inside a sphere of radius $k_c \\propto E_{cut}^{1/2}$). The exact count follows the smooth curve with small lattice-point steps. Pick any key from the table.

In [ ]:
material = "LiCoO2"   # try "Si", "BaTiO3", ...
e = entries[material]
lat = e.reference.structure.lattice
ref_ecut = e.hamiltonian.basis.cutoff_energy_ev

ecuts = np.arange(100, 901, 50)
counts = np.array([num_plane_waves_from_ecut(lat, float(c)) for c in ecuts])
n_ref = num_plane_waves_from_ecut(lat, ref_ecut)
guide = n_ref * (ecuts / ref_ecut) ** 1.5

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(ecuts, counts, "o-", color="#2a9d8f", label="exact count")
ax.plot(ecuts, guide, "--", color="#9aa0a6", label="$\\propto E_{cut}^{3/2}$ through the reference point")
ax.axvline(ref_ecut, color="#e76f51", linestyle=":", label=f"recorded cutoff, {ref_ecut:.0f} eV")
ax.set_xlabel("Cutoff energy (eV)")
ax.set_ylabel("Plane waves at $\\Gamma$")
ax.set_title(f"{material}: basis size vs cutoff")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### What the fingerprint reacts to

`canonical_hash()` is a SHA-256 over the fields that define the calculation: the source record, functional, electron count, basis cutoff and a few more, plus the lattice and atomic positions when the entry records them. Labels such as `tags` are not part of it. Run the cell and check each line against that rule.

In [ ]:
base = entries["Si"]
h0 = base.canonical_hash()

def changed(mutate):
    e = copy.deepcopy(base)
    mutate(e)
    return e.canonical_hash() != h0

checks = {
    "add a tag": changed(lambda e: e.tags.append("reviewed")),
    "raise the cutoff by 50 eV": changed(lambda e: setattr(e.hamiltonian.basis, "cutoff_energy_ev", e.hamiltonian.basis.cutoff_energy_ev + 50)),
    "change the electron count": changed(lambda e: setattr(e.hamiltonian, "num_electrons", e.hamiltonian.num_electrons + 2)),
    "shift one atom by 0.01": changed(lambda e: setattr(
        e.reference.structure.sites[0], "frac_coords",
        tuple(c + 0.01 for c in e.reference.structure.sites[0].frac_coords))),
}
pd.DataFrame({"edit": list(checks), "fingerprint changes": list(checks.values())})

The first row should read `False` and the other three `True`. The fingerprint is not invariant to the choice of unit cell or origin: two descriptions of the same crystal in different cells hash differently, so it identifies *this* setup, not the material in the abstract.

### Round trip through JSON

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "LiCoO2.json"
    write_entry_json(entries["LiCoO2"], path)
    back = read_entry_json(path)
    print(f"{path.name}: {path.stat().st_size:,} bytes")

print("round trip equal:", back == entries["LiCoO2"])
print("same fingerprint:", back.canonical_hash() == entries["LiCoO2"].canonical_hash())

## Part 2. Planning demo with a proxy cost model

The example scripts rank three small candidates with a deliberately simple *proxy* for qubit and T-gate cost. It shows the shape of a selection workflow (rank, Pareto filter, shortlist). It is **not** a resource estimate and does not use the fixtures above, so do not read the recommendation as a scientific result.

### Ensure input artifacts exist

If the CSV files are missing, this cell runs the example scripts to generate them.

In [ ]:
needed = [
    examples_dir / "outputs" / "material_ranking.csv",
    examples_dir / "outputs" / "pareto_candidates.csv",
]

if not all(path.exists() for path in needed):
    print("Generating artifacts from scripts...")
    subprocess.run(
        [sys.executable, str(examples_dir / "material_selection_visual.py")],
        check=True,
        cwd=repo_root,
    )
    subprocess.run(
        [sys.executable, str(examples_dir / "pareto_frontier.py")],
        check=True,
        cwd=repo_root,
    )
else:
    print("All required CSV files already exist.")

for path in needed:
    print(f"- {path} ({path.stat().st_size} bytes)")

In [ ]:
ranking = pd.read_csv(examples_dir / "outputs" / "material_ranking.csv")
pareto = pd.read_csv(examples_dir / "outputs" / "pareto_candidates.csv")

display(ranking)
display(pareto)

### Composite proxy score

In [ ]:
plot_df = ranking.copy()
plot_df["label"] = plot_df["formula"] + " (" + plot_df["source_id"] + ")"
plot_df = plot_df.sort_values("score", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(plot_df["label"], plot_df["score"], color=["#2a9d8f", "#4c78a8", "#e76f51"])
ax.invert_yaxis()
ax.set_title("Proxy score ranking (toy model)")
ax.set_xlabel("Score (lower is better)")
ax.grid(axis="x", linestyle="--", alpha=0.35)

for y, value in enumerate(plot_df["score"]):
    ax.text(value * 1.01, y, f"{value:,.0f}", va="center", fontsize=9)

fig.tight_layout()
plt.show()

### Pareto frontier of the proxies

In [ ]:
pareto["is_pareto_optimal"] = pareto["is_pareto_optimal"].astype(bool)
dom = pareto[~pareto["is_pareto_optimal"]]
front = pareto[pareto["is_pareto_optimal"]].sort_values("qubits_proxy")

fig, ax = plt.subplots(figsize=(8, 5))
if not dom.empty:
    ax.scatter(dom["qubits_proxy"], dom["t_gate_proxy"], s=65, color="#9aa0a6", label="Dominated")

ax.scatter(front["qubits_proxy"], front["t_gate_proxy"], s=85, color="#2a9d8f", label="Pareto-optimal", edgecolors="#1b4332")
if len(front) > 1:
    ax.plot(front["qubits_proxy"], front["t_gate_proxy"], linestyle="--", color="#2a9d8f")

for _, row in pareto.iterrows():
    ax.annotate(row["formula"], (row["qubits_proxy"], row["t_gate_proxy"]), xytext=(5, 5), textcoords="offset points", fontsize=8)

ax.set_title("Pareto frontier: qubits vs T-gate proxy")
ax.set_xlabel("Qubit proxy (lower is better)")
ax.set_ylabel("T-gate proxy (lower is better)")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

### Shortlist

In [ ]:
best = ranking.sort_values("score").iloc[0]
front_list = pareto.loc[pareto["is_pareto_optimal"], ["formula", "source_id"]]

print("Lowest proxy score:")
print(f"- {best['formula']} ({best['source_id']}) with score {best['score']:,.0f}")
print("\nPareto-optimal under the proxies:")
for _, row in front_list.iterrows():
    print(f"- {row['formula']} ({row['source_id']})")

## Next steps

- Fetch your own entries with the Materials Project adapter (`pip install "qmatbridge[mp]"`, then set `MP_API_KEY`); see the README.
- Store entries with `write_entry_json` and share the fingerprint alongside results.
- Exporters to algorithm toolchains are on the roadmap; see the issue tracker.